# REAL ESTATE PRICE PREDICTION -EXPLORATORY Data Analysis (EDA)

**Google Colab copy** — same EDA as `eda.ipynb`, with paths adjusted for Colab.

1. Run the next cell and upload `train.csv` (same file as `data/raw/train.csv` in your project).
2. Or skip upload if you already put `train.csv` in `/content`.


In [ ]:
# Upload train.csv from your computer (Kaggle House Prices training file)
try:
    from google.colab import files
    uploaded = files.upload()  # select train.csv
except ImportError:
    print("Not on Colab — place train.csv in the working directory.")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# Data Loading and Initial Setup

In [ ]:
def load_train_data():
    """Load train.csv from Colab /content after upload, or current directory."""
    from pathlib import Path
    import pandas as pd

    for name in ("train.csv", "/content/train.csv"):
        p = Path(name)
        if p.exists():
            return pd.read_csv(p)
    raise FileNotFoundError(
        "train.csv not found. Run the upload cell above or add train.csv to /content."
    )


# Data Loading and Initial Inspection

In [ ]:
df = load_train_data()

df.head()

In [ ]:
df = load_train_data()

df.head()
df.shape
df.info()
df.describe()

#  features understanding


= number of rows ,useless ID
2  binary feature
3-20 categorical
>50 numeric/ continuous

In [ ]:
pd.DataFrame(df.columns, columns=["Feature Name"])
for col in df.columns:
    print(col, "→", df[col].nunique())

In [ ]:


sns.histplot(df["SalePrice"], kde=True)
plt.title("Target Distribution (SalePrice)")
plt.show()

# missing values


In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
missing

# 📊 Categorical vs Target

In [ ]:
selected_cols = [
    "Neighborhood",
    "HouseStyle",
    "GarageType",
    "SaleCondition",
    "Exterior1st",
    "KitchenQual",
    "HeatingQC",
    "BsmtQual"
]

plt.figure(figsize=(16, 12))

for i, col in enumerate(selected_cols, 1):
    plt.subplot(4, 2, i)
    sns.boxplot(x=df[col], y=df["SalePrice"])
    plt.xticks(rotation=45)
    plt.title(col)

plt.tight_layout()
plt.show()

# Correlation Matrix

In [ ]:
selected_cols = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "TotalBsmtSF",
    "FullBath",
    "YearBuilt",
    "1stFlrSF",
    "2ndFlrSF",
    "SalePrice"
]

# Filter to only columns that exist
selected_cols = [col for col in selected_cols if col in df.columns]

corr = df[selected_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", annot=True, fmt=".2f")
plt.title("Correlation Matrix (Top Features)")
plt.tight_layout()
plt.show()

# detect and remove outliers using IQR method

In [ ]:

Q1 = df["SalePrice"].quantile(0.25)
Q3 = df["SalePrice"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df["SalePrice"] < lower_bound) | (df["SalePrice"] > upper_bound)]
print("Number of outliers:", len(outliers))
df_clean = df[
    (df["SalePrice"] >= lower_bound) &
    (df["SalePrice"] <= upper_bound)
]


# some  feature  engineering
total living space = bigger house bigger price ,
house age = older cheaper,
overall score = quality + condition
bigger gararge = more expensive

In [ ]:
# Feature Engineering Implementation

# Make a proper copy to avoid SettingWithCopyWarning
df_engineered = df_clean.copy()

# Get numeric columns for reference
num_cols = df_engineered.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [col for col in num_cols if col != "SalePrice"]

print("Numeric columns available:", num_cols)

# 1. Total Living Space (proxy: GrLivArea + TotalBsmtSF)
if "GrLivArea" in df_engineered.columns and "TotalBsmtSF" in df_engineered.columns:
    df_engineered["TotalSF"] = df_engineered["GrLivArea"] + df_engineered["TotalBsmtSF"]
    print("✓ Created TotalSF (total living space)")

# 2. House Age (from YearBuilt)
if "YearBuilt" in df_engineered.columns:
    df_engineered["HouseAge"] = 2024 - df_engineered["YearBuilt"]
    print("✓ Created HouseAge")

# 3. Overall Score (quality + condition proxy)
if "OverallQual" in df_engineered.columns and "OverallCond" in df_engineered.columns:
    df_engineered["OverallScore"] = df_engineered["OverallQual"] * df_engineered["OverallCond"]
    print("✓ Created OverallScore (quality * condition)")

# 4. Garage Size Indicator
if "GarageArea" in df_engineered.columns and "GarageCars" in df_engineered.columns:
    df_engineered["GarageScore"] = df_engineered["GarageArea"] * df_engineered["GarageCars"]
    print("✓ Created GarageScore")

print("\nNew engineered features:")
print(df_engineered[["TotalSF", "HouseAge", "OverallScore", "GarageScore"]].head())